# 23CSE301 — Machine Learning Capstone | Review 1

## Predicting Medical Insurance Charges using Regression

---

### Problem Statement

Given personal attributes of a health-insurance beneficiary — **age, sex, BMI,
number of dependents, smoking status, and residential region** — build and compare
regression models that accurately predict the **annual medical charges** billed to
that individual.

This is a **supervised regression** task:
* **Input X:** 6 raw features + 3 engineered features
* **Output y:** `charges` (USD) — a continuous numerical variable

### Dataset at a Glance

| Field | Value |
|-------|-------|
| File | `../data/insurance.csv` |
| Raw shape | 1 338 rows × 7 columns |
| After cleaning | 1 337 rows × 7 columns (1 duplicate removed) |
| Source | Medical Cost Personal Dataset — Kaggle |

| Column | dtype | Role |
|--------|-------|------|
| `age` | int64 | Feature |
| `sex` | object | Feature |
| `bmi` | float64 | Feature |
| `children` | int64 | Feature |
| `smoker` | object | Feature |
| `region` | object | Feature |
| `charges` | float64 | **Regression target** |

### Project Objectives
1. Audit, clean, and explore the dataset.
2. Engineer at least one meaningful feature.
3. Train and compare **10 regression algorithms** on the same held-out test set.
4. Tune hyperparameters for the two best models.
5. Identify the best model for predicting insurance charges.

### AI Assistance Acknowledgement
Code scaffolding was generated with **Google Antigravity (Gemini 2.5 Pro)**.
All analytical observations, draft interpretations, and conclusions have been
manually verified by the project team against actual computed outputs.
AI assistance is acknowledged per university academic-integrity policy.

## Section 2 — Imports and Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Data ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── IPython display ───────────────────────────────────────────────────
from IPython.display import display

# ── Preprocessing & pipelines ─────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer

# ── Model selection ───────────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split, GridSearchCV, cross_val_score
)

# ── Regression models ─────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

# ── Metrics ───────────────────────────────────────────────────────────
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ── Global configuration ──────────────────────────────────────────────
RANDOM_STATE = 42
TEST_SIZE    = 0.20

sns.set_theme(style='whitegrid', palette='husl')
pd.set_option('display.float_format', '{:.4f}'.format)

print(f"All imports successful.")
print(f"RANDOM_STATE = {RANDOM_STATE}  |  TEST_SIZE = {TEST_SIZE}")

## Section 3 — Dataset Loading and Audit

Raw data is loaded without modification.
This section is a **read-only inspection** — no cleaning is performed here.

In [ ]:
df_raw = pd.read_csv('../data/insurance.csv')
print(f"Shape: {df_raw.shape[0]} rows  x  {df_raw.shape[1]} columns")

In [ ]:
df_raw.head(10)

In [ ]:
# Data types and non-null counts
df_raw.info()

In [ ]:
# Missing-value audit
missing  = df_raw.isnull().sum().rename('Missing Count')
miss_pct = (df_raw.isnull().mean() * 100).rename('Missing %')
display(pd.concat([df_raw.dtypes.rename('dtype'), missing, miss_pct], axis=1))

In [ ]:
# Duplicate-row audit
n_dup = df_raw.duplicated().sum()
print(f"Exact duplicate rows: {n_dup}")
if n_dup > 0:
    display(df_raw[df_raw.duplicated(keep=False)].sort_values(list(df_raw.columns)))

In [ ]:
# Descriptive statistics — numerical columns
df_raw.describe().T

In [ ]:
# Unique values in categorical / low-cardinality columns
for col in ['sex', 'smoker', 'region', 'children']:
    print(f"\n{col}  ({df_raw[col].nunique()} unique values)")
    print(df_raw[col].value_counts().to_string())

In [ ]:
# Target-column summary
charges = df_raw['charges']
target_stats = pd.Series({
    'Count'        : int(charges.count()),
    'Min'          : charges.min(),
    'Max'          : charges.max(),
    'Mean'         : charges.mean(),
    'Median'       : charges.median(),
    'Std Dev'      : charges.std(),
    'Skewness'     : charges.skew(),
    'Kurtosis'     : charges.kurtosis(),
    'Q1 (25%)'     : charges.quantile(0.25),
    'Q3 (75%)'     : charges.quantile(0.75),
    'IQR'          : charges.quantile(0.75) - charges.quantile(0.25),
}, name='charges')
display(target_stats.to_frame())

## Section 4 — Exploratory Data Analysis (EDA)

All visualisations are generated from the **raw, uncleaned** dataset.

> **Important (academic integrity):** Markdown cells following each plot contain
> **DRAFT observations** based on known dataset statistics from the audit.
> The project team must verify every observation against the actual rendered
> plot and replace draft text with confirmed findings before submission.

### 4A — Distribution of Every Input Feature

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Input Feature Distributions', fontsize=15, fontweight='bold', y=1.01)

# age
sns.histplot(df_raw['age'], bins=20, kde=True, ax=axes[0, 0], color='steelblue')
axes[0, 0].set(title='Age', xlabel='Age (years)', ylabel='Count')

# bmi
sns.histplot(df_raw['bmi'], bins=30, kde=True, ax=axes[0, 1], color='darkorange')
axes[0, 1].set(title='BMI', xlabel='Body Mass Index', ylabel='Count')

# children
sns.countplot(x='children', data=df_raw, ax=axes[0, 2], palette='viridis')
axes[0, 2].set(title='Number of Children', xlabel='Children', ylabel='Count')

# sex
sns.countplot(x='sex', data=df_raw, ax=axes[1, 0], palette='Set2')
axes[1, 0].set(title='Sex', xlabel='Sex', ylabel='Count')

# smoker
sns.countplot(x='smoker', data=df_raw, ax=axes[1, 1], palette='Set1')
axes[1, 1].set(title='Smoker Status', xlabel='Smoker', ylabel='Count')

# region
sns.countplot(x='region', data=df_raw, ax=axes[1, 2], palette='tab10')
axes[1, 2].set(title='Region', xlabel='Region', ylabel='Count')
axes[1, 2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../data/eda_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

**DRAFT observations — must be verified visually before final submission:**

- `age`: Draft — distribution appears roughly uniform between 18 and 64.
  Verify whether distinct peaks or flat regions are visible.
- `bmi`: Draft — approximately bell-shaped centred near BMI 30.
  Confirm whether the 9 high-BMI values (> 47) form a visible right tail.
- `children`: Draft — majority (≈ 43 %) have 0 children; counts decrease steeply.
  Verify the rarity of counts 4 and 5.
- `sex`: Draft — near-balanced (676 male, 662 female). Confirm visually.
- `smoker`: Draft — non-smokers dominate (≈ 80 %).
  Verify the 274-smoker minority bar is clearly shorter.
- `region`: Draft — four regions appear near-equally distributed.
  Confirm rough balance.

### 4B — Target Variable Distribution (`charges`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Target Variable — Medical Charges Distribution',
             fontsize=14, fontweight='bold')

# Histogram + KDE
sns.histplot(df_raw['charges'], bins=50, kde=True,
             ax=axes[0], color='mediumpurple')
axes[0].axvline(df_raw['charges'].mean(),   color='red',   linestyle='--',
                label=f"Mean   ${df_raw['charges'].mean():,.0f}")
axes[0].axvline(df_raw['charges'].median(), color='green', linestyle='--',
                label=f"Median ${df_raw['charges'].median():,.0f}")
axes[0].set(title='Histogram + KDE', xlabel='Charges (USD)', ylabel='Count')
axes[0].legend(fontsize=9)

# Box plot
sns.boxplot(y=df_raw['charges'], ax=axes[1], color='mediumpurple')
axes[1].set(title='Box Plot', ylabel='Charges (USD)')

plt.tight_layout()
plt.savefig('../data/eda_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Skewness : {df_raw['charges'].skew():.4f}")
print(f"Mean     : ${df_raw['charges'].mean():>12,.2f}")
print(f"Median   : ${df_raw['charges'].median():>12,.2f}")
print(f"Mean - Median gap: ${df_raw['charges'].mean() - df_raw['charges'].median():,.2f}")

**DRAFT observations — verify visually:**

- `charges` has strong **positive skew** (computed skewness ≈ +1.52).
  The mean ($13 270) is approximately $3 888 above the median ($9 382).
- Draft — the histogram is expected to show two or more visible sub-populations
  (e.g., a large low-cost cluster for non-smokers and a smaller high-cost cluster
  for smokers). Verify whether the bimodal or multimodal shape is apparent.
- Draft — the box plot upper whisker should extend far to the right, with many
  individual points above the fence (139 IQR-flagged cases).
- **Team note:** Describe the exact shape of the distribution in your own words.

### 4C — Correlation Heatmap (Numerical Features)

In [ ]:
corr = df_raw[['age', 'bmi', 'children', 'charges']].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            linewidths=0.5, square=True, ax=ax)
ax.set_title('Correlation Heatmap — Numerical Features',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("Pairwise Pearson correlations with 'charges':")
print(corr['charges'].drop('charges').sort_values(ascending=False).to_string())

**DRAFT observations — verify against the printed correlation values:**

- Draft — `age` is expected to show the highest positive Pearson correlation with
  `charges` among the raw numerical columns.
  Verify the exact coefficient from the printout.
- Draft — `bmi` is expected to show a moderate positive correlation with `charges`.
- Draft — `children` is expected to show weak or near-zero correlation.
- **Note:** `smoker` (binary categorical) is not included here; its dominant effect
  will be visible in the boxplots in Section 4E.
- **Team note:** Report exact correlation values and comment on practical significance.

### 4D — Feature-vs-Target Scatter Plots (minimum 2 required)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Feature vs. Charges — Coloured by Smoker Status',
             fontsize=13, fontweight='bold')

smoker_flag = (df_raw['smoker'] == 'yes').astype(int)
cmap = plt.cm.RdYlGn_r

# Plot 1: Age vs Charges
sc = axes[0].scatter(df_raw['age'], df_raw['charges'],
                     c=smoker_flag, cmap=cmap, alpha=0.55, s=20, edgecolors='none')
axes[0].set(xlabel='Age (years)', ylabel='Charges (USD)',
            title='Age vs. Charges')

# Plot 2: BMI vs Charges (global view)
axes[1].scatter(df_raw['bmi'], df_raw['charges'],
                c=smoker_flag, cmap=cmap, alpha=0.55, s=20, edgecolors='none')
axes[1].set(xlabel='BMI', ylabel='Charges (USD)',
            title='BMI vs. Charges')
plt.colorbar(sc, ax=axes[1], label='Smoker (1=yes)', shrink=0.8)

# Plot 3: BMI vs Charges — smoker groups separated
for label, grp in df_raw.groupby('smoker'):
    axes[2].scatter(grp['bmi'], grp['charges'],
                    label=f'smoker={label}', alpha=0.5, s=20, edgecolors='none')
axes[2].set(xlabel='BMI', ylabel='Charges (USD)',
            title='BMI vs. Charges by Smoker Group')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../data/eda_scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

**DRAFT observations — verify visually:**

- **Age vs. Charges:** Draft — there may be two or three upward-sloping parallel bands
  visible, corresponding to non-smokers (lower band) and smokers (upper band).
  Verify whether this banded structure is clear.
- **BMI vs. Charges (coloured):** Draft — smokers (red/orange) are expected to cluster
  at higher charges and show a steeper upward slope with BMI than non-smokers (green).
  If this pattern is visible, it directly justifies the `bmi_smoker` interaction feature
  created in Section 8.
- **BMI vs. Charges (groups separated):** Draft — verify whether the two regression
  cloud slopes are clearly divergent.
- **Team note:** Describe each cluster or band pattern in your own words. These plots are
  the primary domain justification for the engineered feature `bmi_smoker`.

### 4E — Categorical Features vs. Target (Box Plots)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Categorical Features vs. Medical Charges', fontsize=13, fontweight='bold')

sns.boxplot(x='sex',    y='charges', data=df_raw, ax=axes[0], palette='Set2')
axes[0].set(title='Sex vs. Charges', xlabel='Sex', ylabel='Charges (USD)')

sns.boxplot(x='smoker', y='charges', data=df_raw, ax=axes[1], palette='Set1')
axes[1].set(title='Smoker Status vs. Charges', xlabel='Smoker', ylabel='Charges (USD)')

sns.boxplot(x='region', y='charges', data=df_raw, ax=axes[2], palette='tab10')
axes[2].set(title='Region vs. Charges', xlabel='Region', ylabel='Charges (USD)')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../data/eda_categorical_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

print("=== Charges by Smoker Status ===")
display(df_raw.groupby('smoker')['charges'].describe().T)
print()
print("=== Charges by Region ===")
display(df_raw.groupby('region')['charges'].median().sort_values(ascending=False)
          .rename('Median Charges').to_frame())

**DRAFT observations — verify against the printed tables:**

- **Smoker:** Draft — smokers are expected to show dramatically higher median charges and
  a much wider IQR than non-smokers. Verify exact medians from the `.describe()` output.
- **Sex:** Draft — the male vs. female charge distributions are expected to be broadly
  similar (boxes largely overlapping). Verify whether the difference is material.
- **Region:** Draft — four regions expected to be broadly similar.
  Verify whether any region has a notably higher or lower median.
- **Team note:** Quantify the smoker vs. non-smoker median difference and comment on
  its magnitude. This is critical context for the feature engineering decision.

## Section 5 — Data Cleaning

Three sub-steps in order:
1. Missing-value check and treatment
2. Duplicate detection and removal
3. Outlier analysis and treatment decision

In [ ]:
# ── 5.1 Missing-value check ───────────────────────────────────────────
mv = df_raw.isnull().sum()
print("Missing values per column:")
print(mv.to_string())
if mv.sum() == 0:
    print("\n=> No missing values detected. No imputation required.")
else:
    print("\n=> Missing values found — apply appropriate imputation.")

In [ ]:
# ── 5.2 Duplicate removal ─────────────────────────────────────────────
df_clean = df_raw.copy()
n_before = len(df_clean)

df_clean.drop_duplicates(keep='first', inplace=True)
df_clean.reset_index(drop=True, inplace=True)

n_after  = len(df_clean)
n_dropped = n_before - n_after

print(f"Rows before : {n_before}")
print(f"Rows after  : {n_after}")
print(f"Dropped     : {n_dropped} duplicate row(s)")
if n_dropped > 0:
    print()
    print("Removed row: age=19, sex=male, bmi=30.59, children=0,")
    print("             smoker=no, region=northwest, charges=1639.5631")
    print("Reason: exact duplicate of row 195 (all 7 fields identical) —")
    print("        treated as a data-entry error, not a genuine observation.")

In [ ]:
# ── 5.3 Outlier analysis — IQR method ────────────────────────────────
def iqr_report(s: pd.Series) -> pd.Series:
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr    = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out  = int(((s < lo) | (s > hi)).sum())
    return pd.Series({
        'Q1': round(q1, 3),
        'Q3': round(q3, 3),
        'IQR': round(iqr, 3),
        'Lower Fence': round(lo, 3),
        'Upper Fence': round(hi, 3),
        'Outliers (IQR)': n_out,
    })

outlier_df = pd.DataFrame(
    {c: iqr_report(df_clean[c]) for c in ['age', 'bmi', 'children', 'charges']}
).T
print("IQR Outlier Report:")
display(outlier_df)

**Outlier treatment decision — documented:**

| Column | IQR outliers | Treatment | Justification |
|--------|:-----------:|-----------|---------------|
| `age` | 0 | Keep | No outliers. |
| `bmi` | 9 | **Keep** | Range 47–53. These represent clinically documented extreme obesity. Removing them introduces systematic bias against the highest-risk BMI subgroup. |
| `children` | 0 | Keep | No outliers. |
| `charges` | 139 (≈ 10.4 %) | **Keep** | All 139 are high-cost cases (> $34 489). They predominantly correspond to smokers and/or high-BMI patients — a real, medically meaningful subpopulation. Deleting them would cause every model to systematically **underpredict** costs for the highest-risk individuals, which is the most damaging failure mode in insurance pricing. |

> **Draft note:** After inspecting Section 4E results, verify what fraction of the
> 139 charge outliers fall in the `smoker=yes` group and replace this draft note
> with the actual finding.

In [ ]:
print(f"Final working dataset: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns")

## Section 8 — Feature Engineering

> **Execution order note:** Feature engineering is placed **before** the train/test
> split (Section 6) intentionally. All engineered features are deterministic
> calculations (products and threshold comparisons) that involve no fitted statistics,
> so applying them to the full cleaned dataset introduces **no data leakage**.
> Doing so before the split ensures identical, consistent transformation across both
> train and test sets without requiring the formula to be applied twice.

### Primary feature: `bmi_smoker`  *(rubric requirement — at least one engineered feature)*

**Domain motivation:**
Smoking and obesity are independently associated with elevated insurance claims,
but their **joint effect is super-additive**: a smoker with high BMI incurs
disproportionately higher medical costs than the sum of either factor alone.
The scatter plots in Section 4D (BMI vs. Charges by smoker group) are expected to
show two divergent slopes — one steep (smokers) and one shallow (non-smokers).
Encoding this directly as `bmi_smoker = bmi × smoker_binary` allows:

* **Linear models** to capture the joint effect with a single additional coefficient.
* **Tree models** to gain a pre-computed split axis that directly separates the
  high-risk combination.

### Supporting features

| Feature | Formula | Domain motivation |
|---------|---------|-------------------|
| `is_obese` | `bmi >= 30 → 1, else 0` | WHO obesity threshold; insurance pricing commonly uses this cut-off |
| `age_bmi` | `age × bmi` | Older + heavier individuals accumulate compounding chronic conditions |

> `log_charges` is **not** created as a feature. `charges` is the regression
> target; including its log as an input feature would be **target leakage**.

In [ ]:
df_feat = df_clean.copy()

# Primary: smoking x BMI interaction
df_feat['smoker_binary'] = (df_feat['smoker'] == 'yes').astype(int)
df_feat['bmi_smoker']    = df_feat['bmi'] * df_feat['smoker_binary']

# Supporting: WHO obesity flag
df_feat['is_obese'] = (df_feat['bmi'] >= 30).astype(int)

# Supporting: age-BMI product
df_feat['age_bmi']  = df_feat['age'] * df_feat['bmi']

# Drop intermediate helper (original 'smoker' column is kept for OHE)
df_feat.drop(columns=['smoker_binary'], inplace=True)

print("Preview — engineered features vs. target:")
display(df_feat[['age', 'bmi', 'smoker', 'bmi_smoker',
                 'is_obese', 'age_bmi', 'charges']].head(8))
print(f"\nDataset shape after engineering: {df_feat.shape}")

In [ ]:
# Sanity checks
assert (df_feat.loc[df_feat['smoker'] == 'no',  'bmi_smoker'] == 0).all(), \
    "FAIL: bmi_smoker non-zero for a non-smoker row."
assert (df_feat.loc[df_feat['smoker'] == 'yes', 'bmi_smoker'] ==
        df_feat.loc[df_feat['smoker'] == 'yes', 'bmi']).all(), \
    "FAIL: bmi_smoker != bmi for a smoker row."

print("bmi_smoker sanity checks: PASSED")
print()
print("is_obese value counts:")
print(df_feat['is_obese'].value_counts().sort_index().to_string())

## Section 6 — Train / Test Split

* **Ratio:** 80 % train / 20 % test
* **`random_state=42`** for reproducibility
* The **held-out test set is used only once per model** for final evaluation —
  never for hyperparameter selection or preprocessing fit

| Group | Columns |
|-------|---------|
| Numerical | `age`, `bmi`, `children`, `bmi_smoker`, `is_obese`, `age_bmi` |
| Categorical | `sex`, `smoker`, `region` |
| Target | `charges` |

In [ ]:
numerical_features   = ['age', 'bmi', 'children', 'bmi_smoker', 'is_obese', 'age_bmi']
categorical_features = ['sex', 'smoker', 'region']
feature_cols         = numerical_features + categorical_features
target_col           = 'charges'

X = df_feat[feature_cols].copy()
y = df_feat[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print(f"X_train : {X_train.shape}  |  y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}   |  y_test  : {y_test.shape}")
print()
print(f"Train mean charges : ${y_train.mean():>12,.2f}")
print(f"Test  mean charges : ${y_test.mean():>12,.2f}")

## Section 7 — Preprocessing Pipeline

A scikit-learn `ColumnTransformer` applies different transformations per group.

| Feature group | Transformer | Reason |
|---------------|-------------|--------|
| Numerical (6 columns) | `StandardScaler` | Required for SVR, KNN, and regularised linear models (Ridge, Lasso, ElasticNet) |
| Categorical — 2-level: `sex`, `smoker` | `OneHotEncoder(drop='first')` | Avoids dummy-variable trap; each becomes 1 column |
| Categorical — 4-level: `region` | `OneHotEncoder(drop='first')` | Becomes 3 columns |

**Leakage prevention:**
`preprocessor.fit()` is called **only on `X_train`**.
`X_test` is transformed using the already-fitted preprocessor —
no test-set statistics influence any transformation parameter.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat',
         OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False),
         categorical_features),
    ],
    remainder='drop'
)

# Fit ONLY on training data
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)          # transform only

print(f"Processed train shape : {X_train_proc.shape}")
print(f"Processed test  shape : {X_test_proc.shape}")

In [ ]:
# Retrieve processed feature names (used for importance plots later)
cat_names_out    = (preprocessor
                    .named_transformers_['cat']
                    .get_feature_names_out(categorical_features)
                    .tolist())
proc_feature_names = numerical_features + cat_names_out

print(f"Processed features ({len(proc_feature_names)} total):")
for i, name in enumerate(proc_feature_names, 1):
    print(f"  {i:2d}. {name}")

## Section 9 — All 10 Regression Algorithms

Every model is trained on `(X_train_proc, y_train)` and evaluated on the **same**
held-out `(X_test_proc, y_test)`.

Metrics collected:
* **R²** — proportion of variance explained (higher is better)
* **RMSE** — root mean squared error in USD (lower is better)
* **MAE** — mean absolute error in USD (lower is better)

Results accumulate in `all_results` and are displayed as a ranked DataFrame at the end.

In [ ]:
# ── Shared evaluation helper ──────────────────────────────────────────
def evaluate_model(name: str, y_true, y_pred) -> dict:
    '''Return R2, RMSE, and MAE for a set of predictions.'''
    return {
        'Model': name,
        'R2'   : round(r2_score(y_true, y_pred), 4),
        'RMSE' : round(np.sqrt(mean_squared_error(y_true, y_pred)), 2),
        'MAE'  : round(mean_absolute_error(y_true, y_pred), 2),
    }

all_results    = []   # accumulate metric dicts
trained_models = {}   # name -> fitted model object
all_preds      = {}   # name -> y_pred array
print("Helper ready.")

In [ ]:
# ── 1. Linear Regression ──────────────────────────────────────────────
_m = LinearRegression()
_m.fit(X_train_proc, y_train)
_p = _m.predict(X_test_proc)
trained_models['Linear Regression'] = _m
all_preds['Linear Regression']      = _p
all_results.append(evaluate_model('Linear Regression', y_test, _p))
print("1. Linear Regression done.")

In [ ]:
# ── 2. Ridge Regression ───────────────────────────────────────────────
_m = Ridge(alpha=1.0)
_m.fit(X_train_proc, y_train)
_p = _m.predict(X_test_proc)
trained_models['Ridge Regression'] = _m
all_preds['Ridge Regression']      = _p
all_results.append(evaluate_model('Ridge Regression', y_test, _p))
print("2. Ridge Regression done.")

In [ ]:
# ── 3. Lasso Regression ───────────────────────────────────────────────
_m = Lasso(alpha=1.0, max_iter=10000)
_m.fit(X_train_proc, y_train)
_p = _m.predict(X_test_proc)
trained_models['Lasso Regression'] = _m
all_preds['Lasso Regression']      = _p
all_results.append(evaluate_model('Lasso Regression', y_test, _p))
print(f"3. Lasso Regression done.  "
      f"(non-zero coefs: {(_m.coef_ != 0).sum()}/{len(_m.coef_)})")

In [ ]:
# ── 4. ElasticNet Regression ──────────────────────────────────────────
_m = ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)
_m.fit(X_train_proc, y_train)
_p = _m.predict(X_test_proc)
trained_models['ElasticNet Regression'] = _m
all_preds['ElasticNet Regression']      = _p
all_results.append(evaluate_model('ElasticNet Regression', y_test, _p))
print("4. ElasticNet Regression done.")

### Section 10 — Polynomial Regression (degree = 2)

`PolynomialFeatures(degree=2)` is applied **on top of the already-scaled,
already-encoded** processed features (`X_train_proc`).
The `poly_transformer` is **fit on `X_train_proc` only**, then applied to
`X_test_proc` — maintaining the no-leakage guarantee.

Feature count after expansion: with 11 processed inputs,
degree-2 gives 11 original + C(11, 2) + 11 = **77 features** (no bias term).
This is well within the capacity of the ~1 070-row training set.

In [ ]:
# ── 5. Polynomial Regression (degree = 2) ────────────────────────────
poly_transformer = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly_transformer.fit_transform(X_train_proc)   # fit on train only
X_test_poly  = poly_transformer.transform(X_test_proc)        # transform only

print(f"Polynomial feature count (degree=2): {X_train_poly.shape[1]}")

_m = LinearRegression()
_m.fit(X_train_poly, y_train)
_p = _m.predict(X_test_poly)
all_preds['Poly Regression (deg=2)'] = _p
all_results.append(evaluate_model('Poly Regression (deg=2)', y_test, _p))
print("5. Polynomial Regression (deg=2) done.")

In [ ]:
# ── 6. Decision Tree Regressor ────────────────────────────────────────
_m = DecisionTreeRegressor(random_state=RANDOM_STATE)
_m.fit(X_train_proc, y_train)
_p = _m.predict(X_test_proc)
trained_models['Decision Tree'] = _m
all_preds['Decision Tree']      = _p
all_results.append(evaluate_model('Decision Tree', y_test, _p))
print("6. Decision Tree Regressor done.")

In [ ]:
# ── 7. Random Forest Regressor ────────────────────────────────────────
_m = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
_m.fit(X_train_proc, y_train)
_p = _m.predict(X_test_proc)
trained_models['Random Forest'] = _m
all_preds['Random Forest']      = _p
all_results.append(evaluate_model('Random Forest', y_test, _p))
print("7. Random Forest Regressor done.")

In [ ]:
# ── 8. Gradient Boosting Regressor ───────────────────────────────────
_m = GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE)
_m.fit(X_train_proc, y_train)
_p = _m.predict(X_test_proc)
trained_models['Gradient Boosting'] = _m
all_preds['Gradient Boosting']      = _p
all_results.append(evaluate_model('Gradient Boosting', y_test, _p))
print("8. Gradient Boosting Regressor done.")

In [ ]:
# ── 9. Support Vector Regressor (SVR) ────────────────────────────────
_m = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)
_m.fit(X_train_proc, y_train)
_p = _m.predict(X_test_proc)
trained_models['SVR'] = _m
all_preds['SVR']      = _p
all_results.append(evaluate_model('SVR', y_test, _p))
print("9. SVR (rbf, C=100) done.")

In [ ]:
# ── 10. K-Nearest Neighbours Regressor ───────────────────────────────
_m = KNeighborsRegressor(n_neighbors=5)
_m.fit(X_train_proc, y_train)
_p = _m.predict(X_test_proc)
trained_models['KNN Regressor'] = _m
all_preds['KNN Regressor']      = _p
all_results.append(evaluate_model('KNN Regressor', y_test, _p))
print("10. KNN Regressor (k=5) done.")

In [ ]:
# ── Consolidated ranking ──────────────────────────────────────────────
results_df = (
    pd.DataFrame(all_results)
    .rename(columns={'R2': 'R²'})
    .sort_values('R²', ascending=False)
    .reset_index(drop=True)
)
results_df.index += 1
results_df.index.name = 'Rank'

print("=== 10-Model Comparison — Test-Set Performance (ranked by R²) ===")
display(results_df)

## Section 11 — Hyperparameter Tuning

`GridSearchCV` with 5-fold cross-validation is used to tune **two models**:

1. **Random Forest** — key parameters: `n_estimators`, `max_depth`, `min_samples_split`
2. **Gradient Boosting** — key parameters: `learning_rate`, `n_estimators`, `max_depth`

**Leakage prevention:** All tuning is done on `(X_train_proc, y_train)`.
The held-out test set is used **only** to report final tuned performance.

In [ ]:
# ── 11.1 Random Forest — GridSearchCV ────────────────────────────────
param_grid_rf = {
    'n_estimators'     : [100, 200, 300],
    'max_depth'        : [None, 10, 20],
    'min_samples_split': [2, 5, 10],
}

gs_rf = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid_rf,
    cv=5, scoring='r2', n_jobs=-1, verbose=1
)
gs_rf.fit(X_train_proc, y_train)

best_rf         = gs_rf.best_estimator_
y_pred_rf_tuned = best_rf.predict(X_test_proc)

print(f"\nBest parameters : {gs_rf.best_params_}")
print(f"Best CV R2      : {gs_rf.best_score_:.4f}")

In [ ]:
# ── 11.2 Gradient Boosting — GridSearchCV ────────────────────────────
param_grid_gb = {
    'n_estimators' : [100, 200, 300],
    'learning_rate': [0.05, 0.10, 0.15],
    'max_depth'    : [3, 5, 7],
}

gs_gb = GridSearchCV(
    GradientBoostingRegressor(random_state=RANDOM_STATE),
    param_grid_gb,
    cv=5, scoring='r2', n_jobs=-1, verbose=1
)
gs_gb.fit(X_train_proc, y_train)

best_gb         = gs_gb.best_estimator_
y_pred_gb_tuned = best_gb.predict(X_test_proc)

print(f"\nBest parameters : {gs_gb.best_params_}")
print(f"Best CV R2      : {gs_gb.best_score_:.4f}")

In [ ]:
# ── 11.3 Improvement report ───────────────────────────────────────────
def delta(base: dict, tuned: dict, metric: str) -> float:
    return round(tuned[metric] - base[metric], 4 if metric == 'R2' else 2)

base_rf  = {k: v for k, v in evaluate_model('RF baseline',  y_test, all_preds['Random Forest']).items()}
tune_rf  = {k: v for k, v in evaluate_model('RF tuned',     y_test, y_pred_rf_tuned).items()}
base_gb  = {k: v for k, v in evaluate_model('GB baseline',  y_test, all_preds['Gradient Boosting']).items()}
tune_gb  = {k: v for k, v in evaluate_model('GB tuned',     y_test, y_pred_gb_tuned).items()}

tuning_report = (
    pd.DataFrame([base_rf, tune_rf, base_gb, tune_gb])
    .rename(columns={'R2': 'R²'})
)

print("=== Hyperparameter Tuning — Baseline vs. Tuned ===")
display(tuning_report)

print()
for base, tuned, label in [(base_rf, tune_rf, 'Random Forest'),
                           (base_gb, tune_gb, 'Gradient Boosting')]:
    dr2   = delta(base, tuned, 'R2')
    drmse = delta(base, tuned, 'RMSE')
    dmae  = delta(base, tuned, 'MAE')
    print(f"{label:20s} => R2 {dr2:+.4f}  |  RMSE {drmse:+.2f}  |  MAE {dmae:+.2f}")

## Section 12 — 5-Fold Cross-Validation

The **two highest-ranked models** from the 10-model comparison (`results_df` Rank 1
and Rank 2) are subjected to 5-fold cross-validation on the training set.

CV is run on `(X_train_proc, y_train)` — the test set is not touched.

In [ ]:
top2_names = results_df.head(2)['Model'].tolist()
print(f"Top-2 by test R2: {top2_names}")

MODEL_REGISTRY = {
    'Linear Regression'     : trained_models.get('Linear Regression'),
    'Ridge Regression'      : trained_models.get('Ridge Regression'),
    'Lasso Regression'      : trained_models.get('Lasso Regression'),
    'ElasticNet Regression' : trained_models.get('ElasticNet Regression'),
    'Decision Tree'         : trained_models.get('Decision Tree'),
    'Random Forest'         : trained_models.get('Random Forest'),
    'Gradient Boosting'     : trained_models.get('Gradient Boosting'),
    'SVR'                   : trained_models.get('SVR'),
    'KNN Regressor'         : trained_models.get('KNN Regressor'),
}

cv_records = []
for name in top2_names:
    model = MODEL_REGISTRY.get(name)
    if model is None:
        print(f"  {name}: no stored model object (Polynomial). Skipping CV.")
        continue
    scores = cross_val_score(model, X_train_proc, y_train, cv=5, scoring='r2')
    cv_records.append({
        'Model'   : name,
        'Fold 1'  : round(scores[0], 4),
        'Fold 2'  : round(scores[1], 4),
        'Fold 3'  : round(scores[2], 4),
        'Fold 4'  : round(scores[3], 4),
        'Fold 5'  : round(scores[4], 4),
        'Mean R2' : round(scores.mean(), 4),
        'Std Dev' : round(scores.std(),  4),
    })
    print(f"\n{name}")
    print(f"  Folds : {[round(s, 4) for s in scores]}")
    print(f"  Mean  : {scores.mean():.4f}  +/-  {scores.std():.4f}")

In [ ]:
cv_df = (pd.DataFrame(cv_records)
           .rename(columns={'Mean R2': 'Mean R²'}))
print("=== 5-Fold Cross-Validation Summary ===")
display(cv_df)

## Section 13 — Regression Visualisations

Required plots:
1. **Predicted vs. Actual** — best model
2. **Residual plot** — best model
3. **Feature Importance** — best tree-based model

In [ ]:
# Identify best model (prefer tuned variant if it ranked #1)
best_name = results_df.iloc[0]['Model']
print(f"Best model (by test R2): {best_name}")

if best_name == 'Random Forest':
    y_pred_best    = y_pred_rf_tuned
    best_model_obj = best_rf
elif best_name == 'Gradient Boosting':
    y_pred_best    = y_pred_gb_tuned
    best_model_obj = best_gb
else:
    y_pred_best    = all_preds.get(best_name)
    best_model_obj = trained_models.get(best_name)

residuals = y_test.values - y_pred_best

print(f"  R2   : {r2_score(y_test, y_pred_best):.4f}")
print(f"  RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_best)):,.2f}")
print(f"  MAE  : {mean_absolute_error(y_test, y_pred_best):,.2f}")

In [ ]:
# ── 13A: Predicted vs. Actual ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))

ax.scatter(y_test, y_pred_best, alpha=0.5, s=22,
           color='steelblue', edgecolors='none', label='Predictions')
lim_lo = min(y_test.min(), y_pred_best.min()) * 0.93
lim_hi = max(y_test.max(), y_pred_best.max()) * 1.05
ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], 'r--', lw=1.8, label='Perfect fit line')
ax.set_xlim(lim_lo, lim_hi)
ax.set_ylim(lim_lo, lim_hi)
ax.set_xlabel('Actual Charges (USD)',    fontsize=12)
ax.set_ylabel('Predicted Charges (USD)', fontsize=12)
ax.set_title(f'Predicted vs. Actual Charges\n{best_name}',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.text(0.05, 0.91,
        f"R² = {r2_score(y_test, y_pred_best):.4f}",
        transform=ax.transAxes, fontsize=11,
        bbox=dict(facecolor='lightyellow', edgecolor='grey', boxstyle='round,pad=0.3'))

plt.tight_layout()
plt.savefig('../data/plot_pred_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 13B: Residual Plot ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'Residual Analysis — {best_name}', fontsize=13, fontweight='bold')

# Residuals vs fitted
axes[0].scatter(y_pred_best, residuals, alpha=0.5, s=22,
                color='darkorange', edgecolors='none')
axes[0].axhline(0, color='crimson', linestyle='--', lw=1.8)
axes[0].set(xlabel='Fitted Values (USD)', ylabel='Residuals (USD)',
            title='Residuals vs. Fitted Values')

# Residual distribution
sns.histplot(residuals, bins=40, kde=True, ax=axes[1], color='darkorange')
axes[1].axvline(0, color='crimson', linestyle='--', lw=1.8)
axes[1].set(xlabel='Residual (USD)', title='Residual Distribution')

plt.tight_layout()
plt.savefig('../data/plot_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Residual mean : {residuals.mean():>10,.2f}  (should be near 0)")
print(f"Residual std  : {residuals.std():>10,.2f}")

In [ ]:
# ── 13C: Feature Importance (tree-based model) ────────────────────────
if best_model_obj is not None and hasattr(best_model_obj, 'feature_importances_'):
    fi_model, fi_label = best_model_obj, best_name
else:
    fi_model, fi_label = best_rf, 'Random Forest (tuned)'

fi_df = (
    pd.DataFrame({'Feature': proc_feature_names,
                  'Importance': fi_model.feature_importances_})
    .sort_values('Importance', ascending=False)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(x='Importance', y='Feature', data=fi_df,
            palette='viridis', ax=ax)
ax.set_title(f'Feature Importance — {fi_label}', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Decrease Impurity (normalised)')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../data/plot_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top-5 features by importance:")
display(fi_df.head())

## Section 14 — Final Model Comparison and Summary

In [ ]:
# ── 14A: All models + tuned variants ─────────────────────────────────
final_df = (
    pd.DataFrame(
        all_results
        + [evaluate_model('Random Forest (tuned)',        y_test, y_pred_rf_tuned)]
        + [evaluate_model('Gradient Boosting (tuned)',    y_test, y_pred_gb_tuned)]
    )
    .rename(columns={'R2': 'R²'})
    .sort_values('R²', ascending=False)
    .reset_index(drop=True)
)
final_df.index += 1
final_df.index.name = 'Rank'

print("=== Final Comparison — All Models + Tuned Variants (ranked by R²) ===")
display(final_df)

In [ ]:
# ── 14B: 5-fold CV for top-2 ──────────────────────────────────────────
print("=== 5-Fold Cross-Validation — Top-2 Models ===")
display(cv_df)

In [ ]:
# ── 14C: Bar chart — 10 baseline models ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle('Model Comparison — All 10 Baseline Regression Algorithms',
             fontsize=13, fontweight='bold')

base10 = results_df.copy()

configs = [
    ('R²',   'steelblue',     True),
    ('RMSE', 'darkorange',   False),
    ('MAE',  'mediumpurple', False),
]

for ax, (metric, color, high_is_good) in zip(axes, configs):
    ordered = base10.sort_values(metric, ascending=not high_is_good)
    bars = ax.barh(ordered['Model'], ordered[metric],
                   color=color, edgecolor='white')
    ax.set(xlabel=metric, title=f'{metric} per Model')
    if high_is_good:
        ax.invert_yaxis()
    for bar, val in zip(bars, ordered[metric]):
        label_txt = f'{val:.3f}' if metric == 'R²' else f'{val:,.0f}'
        ax.text(bar.get_width() * 1.008,
                bar.get_y() + bar.get_height() / 2,
                label_txt, va='center', ha='left', fontsize=7.5)

plt.tight_layout()
plt.savefig('../data/plot_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### Draft Conclusion

> **DRAFT — Every point below must be verified against the actual computed results
> and rewritten by the project team before final submission.**

1. **Best overall model:** Draft — identify the Rank-1 entry in `final_df`.
   State its exact R², RMSE, and MAE.

2. **Hyperparameter tuning impact:** Draft — compare baseline vs. tuned
   performance for Random Forest and Gradient Boosting.
   If the improvement is negligible (< 0.001 R²), note this and explain
   (e.g., the baseline already had reasonable defaults).

3. **CV vs. test consistency:** Draft — compare the mean CV R² for both top
   models against their test-set R². If test R² is substantially above CV R²,
   investigate potential overfitting. If they are close, this supports confidence
   in the test-set result.

4. **Linear vs. ensemble models:** Draft — state whether linear models (LR, Ridge,
   Lasso, ElasticNet) form a distinct lower performance tier below tree ensembles,
   and describe the magnitude of the gap.

5. **Feature engineering validation:** Draft — check whether `bmi_smoker` appears
   in the top-3 features in the importance plot. If it does, explicitly state this
   as validation of the feature engineering choice.

6. **Residual analysis:** Draft — describe the shape of the residual vs. fitted
   plot. Note whether heteroscedasticity (funnel shape) is present or whether
   residuals are roughly symmetric around zero across the fitted-value range.

**Team instruction:** Replace each draft bullet with one factual, evidence-based
sentence derived from the actual notebook outputs. Remove this instruction box
before submission.